# Tesla Model 3 Performance — telemetry video overlay

This notebook is dedicated to designing, synchronizing, previewing, and rendering a telemetry HUD over Tesla Track Mode video. Heavy processing lives in `telemetry_overlay.py`; the notebook provides configuration, diagnostics, interactive still previews, and guarded video renders.

The default HUD includes speed, current and best lap time, live delta at the same track position, track position, throttle, brake, regenerative power, signed power, numerical and graphical g-force, normalized four-corner tyre slip, state of charge, battery thermal load, and front/rear inverter thermal load. Tesla's export does **not** contain actual motor temperature or temperature in °C, so inverter thermal indicators are labelled accurately.

## Environment

Run this notebook with the repository's `car_track` Conda environment. Install or refresh dependencies from the repository root with:

```powershell
C:\Users\11320\anaconda3\envs\car_track\python.exe -m pip install -r requirements.txt
```

In [1]:
from pathlib import Path
import sys

EXPECTED_PYTHON = Path(r"C:\Users\11320\anaconda3\envs\car_track\python.exe")
if Path(sys.executable).resolve() != EXPECTED_PYTHON.resolve():
    raise RuntimeError(
        f"Wrong notebook interpreter: {sys.executable}\n"
        f"Select the car_track kernel at {EXPECTED_PYTHON}."
    )

import ipywidgets as widgets
import pandas as pd
from IPython.display import Video, clear_output, display

from telemetry_overlay import (
    OverlayStyle,
    TelemetryOverlayRenderer,
    TelemetrySampler,
    attach_session_clock,
    build_sync_info,
    build_track_reference,
    discover_recording_pairs,
    format_overlay_time,
    load_overlay_telemetry,
    probe_video,
    recording_summary,
    render_overlay_preview,
    render_overlay_video,
    select_recording,
)

print(f"Using {sys.executable}")

Using c:\Users\11320\anaconda3\envs\car_track\python.exe


## Configuration

`DEFAULT_SESSION` uses the telemetry start timestamp in `YYYYMMDD_HHMMSS` form. This notebook is configured for the final recording on 2026-07-14. The same recording contains timed laps and is used to build its reference track.

Just change both DEFAULT_SESSION and REFERENCE_SESSION to the corresponding video/session you want

In [2]:
DATA_DIR = Path("data/20260714")
DEFAULT_SESSION = "20260714_192847"
REFERENCE_SESSION = "20260714_192847"
REFERENCE_LAP = None  # None chooses the fastest lap near the median GPS distance.

PREVIEW_TIME_S = 450.0
SYNC_ADJUST_S = 0.0  # Positive values select later telemetry for the same video frame.
PREVIEW_WIDTH = 1440
PREVIEW_CLIP_DURATION_S = 10.0
OUTPUT_DIR = Path("outputs/20260714")

STYLE = OverlayStyle(
    speed_unit="km/h",
    max_regen_kw=200.0,
    show_tire_pressures=True,
    show_estimated_temperatures=False,
)

## Discover, pair, and synchronize recordings

CSV and MP4 timestamps differ by up to several seconds, so files are paired by nearest unique filename timestamp. A monotonic telemetry clock is then fitted using row count, video duration, and the filename offset. Every CSV row is retained because duplicate rows still represent sampling time.

In [3]:
recordings = discover_recording_pairs(DATA_DIR)
display(recording_summary(recordings))

recording = select_recording(recordings, DEFAULT_SESSION)
video_info = probe_video(recording.video_path)
telemetry = load_overlay_telemetry(recording.telemetry_path)
sync = build_sync_info(
    telemetry,
    video_info,
    video_start_offset_s=recording.filename_offset_s,
)
telemetry_with_clock = attach_session_clock(telemetry, sync)

reference_recording = select_recording(recordings, REFERENCE_SESSION)
if reference_recording.telemetry_path == recording.telemetry_path:
    reference_telemetry = telemetry
else:
    reference_telemetry = load_overlay_telemetry(reference_recording.telemetry_path)
track = build_track_reference(reference_telemetry, lap=REFERENCE_LAP)
sampler = TelemetrySampler(telemetry, sync)
renderer = TelemetryOverlayRenderer(track, STYLE)

diagnostics = pd.DataFrame(
    {
        "Metric": [
            "Session", "Telemetry rows", "Video duration (s)",
            "Video resolution", "Average video fps",
            "Filename offset (s)", "Fitted telemetry Hz",
            "Lap-derived telemetry Hz", "Rate source",
            "Reference lap", "Reference lap time", "Reference distance (m)",
            "Tyre slip baseline (FL/FR/RL/RR %)",
            "Tyre slip deadband (%)", "Tyre slip robust scale (%)",
        ],
        "Value": [
            recording.session_id, len(telemetry), f"{video_info.duration_s:.3f}",
            f"{video_info.width}×{video_info.height}", f"{video_info.average_rate:.3f}",
            f"{recording.filename_offset_s:+.3f}", f"{sync.sample_rate_hz:.4f}",
            "—" if sync.lap_rate_hz is None else f"{sync.lap_rate_hz:.4f}",
            sync.rate_source, track.lap, format_overlay_time(track.lap_time_ms),
            f"{track.total_distance_m:.1f}",
            " / ".join(
                "--" if pd.isna(value) else f"{value:.2f}"
                for value in sampler.tire_slip_baseline_pct
            ),
            f"{sampler.tire_slip_deadband_pct:.2f}",
            f"{sampler.tire_slip_scale_pct:.2f}",
        ],
    }
)
display(diagnostics)

,Session,Telemetry CSV,Video,Filename offset (s),Thumbnail
0,20260714_182812,telemetry-v1-2026-07-14-18_28_12.csv,laps-2026-07-14-18_28_13.mp4,1.0,laps-2026-07-14-18_28_13-thumb.png
1,20260714_184410,telemetry-v1-2026-07-14-18_44_10.csv,laps-2026-07-14-18_44_11.mp4,1.0,laps-2026-07-14-18_44_11-thumb.png
2,20260714_192842,telemetry-v1-2026-07-14-19_28_42.csv,laps-2026-07-14-19_28_44.mp4,2.0,laps-2026-07-14-19_28_44-thumb.png
3,20260714_192847,telemetry-v1-2026-07-14-19_28_47.csv,laps-2026-07-14-19_28_47.mp4,0.0,laps-2026-07-14-19_28_47-thumb.png
4,20260714_201959,telemetry-v1-2026-07-14-20_19_59.csv,laps-2026-07-14-20_20_00.mp4,1.0,laps-2026-07-14-20_20_00-thumb.png
5,20260714_205823,telemetry-v1-2026-07-14-20_58_23.csv,laps-2026-07-14-20_58_23.mp4,0.0,laps-2026-07-14-20_58_23-thumb.png
6,20260714_210040,telemetry-v1-2026-07-14-21_00_40.csv,laps-2026-07-14-21_00_42.mp4,2.0,laps-2026-07-14-21_00_42-thumb.png
7,20260714_213600,telemetry-v1-2026-07-14-21_36_00.csv,laps-2026-07-14-21_36_04.mp4,4.0,laps-2026-07-14-21_36_04-thumb.png
8,20260714_214053,telemetry-v1-2026-07-14-21_40_53.csv,laps-2026-07-14-21_40_55.mp4,2.0,laps-2026-07-14-21_40_55-thumb.png


,Metric,Value
0,Session,20260714_192847
1,Telemetry rows,46705
2,Video duration (s),934.200
3,Video resolution,2896×1876
4,Average video fps,36.001
5,Filename offset (s),+0.000
6,Fitted telemetry Hz,49.9936
7,Lap-derived telemetry Hz,49.9930
8,Rate source,video-end alignment
9,Reference lap,2


## Interactive still preview and sync tuning

Choose a video timestamp and synchronization adjustment, then click **Render still preview**. Check a clear throttle, brake, steering, or acceleration event against the image. Positive synchronization adjustment uses later telemetry. Once the alignment is correct, copy the slider value back to `SYNC_ADJUST_S`.

In [4]:
preview_time_slider = widgets.FloatSlider(
    value=min(PREVIEW_TIME_S, max(0.0, video_info.duration_s - 0.1)),
    min=0.0,
    max=max(0.1, video_info.duration_s - 0.05),
    step=0.1,
    description="Video time",
    readout_format=".1f",
    layout=widgets.Layout(width="760px"),
)
sync_adjust_slider = widgets.FloatSlider(
    value=SYNC_ADJUST_S,
    min=-2.0,
    max=2.0,
    step=0.05,
    description="Sync adjust",
    readout_format="+.2f",
    layout=widgets.Layout(width="760px"),
)
preview_button = widgets.Button(description="Render still preview", icon="image")
preview_output = widgets.Output()

def show_overlay_preview(_=None):
    with preview_output:
        clear_output(wait=True)
        result = render_overlay_preview(
            recording.video_path,
            sampler,
            renderer,
            video_time_s=preview_time_slider.value,
            sync_adjust_s=sync_adjust_slider.value,
            output_width=PREVIEW_WIDTH,
        )
        display(result.image)
        print(
            f"Requested {result.requested_time_s:.3f}s; decoded {result.actual_time_s:.3f}s; "
            f"Lap {result.sample.lap}; speed {result.sample.speed_kmh:.1f} km/h; "
            f"sync adjustment {sync_adjust_slider.value:+.2f}s"
        )

preview_button.on_click(show_overlay_preview)
display(widgets.VBox([preview_time_slider, sync_adjust_slider, preview_button, preview_output]))

## Render a short preview clip

Set the switch to `True` only after checking the still preview. The preview is downscaled for quicker iteration and written below `outputs/`, which is ignored by Git.

In [5]:
RENDER_PREVIEW_CLIP = True

if RENDER_PREVIEW_CLIP:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    progress_bar = widgets.FloatProgress(min=0, max=1, description="Rendering")
    progress_label = widgets.Label()
    display(widgets.HBox([progress_bar, progress_label]))

    def update_progress(frames, current_time, fraction):
        if fraction is not None:
            progress_bar.value = fraction
        progress_label.value = f"{frames:,} frames · video {current_time:.1f}s"

    preview_path = OUTPUT_DIR / f"{recording.session_id}-overlay-preview.mp4"
    preview_render = render_overlay_video(
        recording.video_path, sampler, renderer, preview_path,
        start_s=preview_time_slider.value,
        duration_s=PREVIEW_CLIP_DURATION_S,
        sync_adjust_s=sync_adjust_slider.value,
        output_width=PREVIEW_WIDTH,
        preset="veryfast",
        progress=update_progress,
    )
    progress_bar.value = 1
    display(preview_render)
    display(Video(filename=str(preview_path), embed=False))
else:
    print("Set RENDER_PREVIEW_CLIP = True to encode a short preview.")

RenderResult(output_path=WindowsPath('outputs/20260714/20260714_192847-overlay-preview.mp4'), frames=360, start_time_s=450.0155, end_time_s=459.9883, width=1440, height=934)

## Render the complete recording

Complete recordings can take substantial time to encode. Keep this final action explicit. The configured 1440-pixel output preserves the source aspect ratio while keeping render time and file size practical. The renderer uses native video scaling, cached static HUD layers, precomputed telemetry interpolation, streaming frame processing, source presentation timestamps for telemetry sampling, and a stable constant-frame-rate MP4 output.

In [6]:
RENDER_FULL_VIDEO = True
FULL_OUTPUT_WIDTH = 1440  # Produces 1440×934 while preserving the source aspect ratio.

if RENDER_FULL_VIDEO:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    full_progress = widgets.FloatProgress(min=0, max=1, description="Rendering")
    full_label = widgets.Label()
    display(widgets.HBox([full_progress, full_label]))

    def update_full_progress(frames, current_time, fraction):
        if fraction is not None:
            full_progress.value = fraction
        full_label.value = f"{frames:,} frames · video {current_time:.1f}s"

    full_path = OUTPUT_DIR / f"{recording.session_id}-overlay.mp4"
    full_render = render_overlay_video(
        recording.video_path, sampler, renderer, full_path,
        sync_adjust_s=sync_adjust_slider.value,
        output_width=FULL_OUTPUT_WIDTH,
        crf=20,
        preset="veryfast",
        progress=update_full_progress,
    )
    full_progress.value = 1
    display(full_render)
    display(Video(filename=str(full_path), embed=False))
else:
    print("Set RENDER_FULL_VIDEO = True only after validating the preview clip.")

RenderResult(output_path=WindowsPath('outputs/20260714/20260714_192847-overlay.mp4'), frames=33631, start_time_s=0.0, end_time_s=934.1759, width=1440, height=934)

## Data interpretation notes

- `Battery Temp (%)`, `Front Inverter Temp (%)`, and `Rear Inverter Temp (%)` are normalized thermal indicators. They are multiplied by 100 for display and must not be labelled as °C.
- The export has no true motor winding/rotor temperature channel. Inverter temperature is shown instead and labelled explicitly.
- Negative brake-pressure offsets are retained in source data and clamped to zero only in the HUD.
- Tesla's signed `Power Level (KW)` is shown directly in the power panel. Negative power is also converted to a positive regen amount with `max(0, -power_kw)` and a 200 kW bar scale.
- `G` is the resultant planar acceleration `hypot(lateral_g, longitudinal_g)`; the lateral and longitudinal signed values remain visible below it.
- `BEST` is the quickest distance-consistent reference lap. `DELTA` compares current elapsed time with that lap's elapsed time at the same projected track position; green is ahead and red is behind.
- Zero tyre pressure is treated as unavailable. Estimated brake temperature remains optional because it is quantized.
- The tyre-slip numbers preserve Tesla's signed estimates. Bars use the magnitude above each tyre's straight-line baseline, remove one quantization step as deadband, normalize all four tyres to a shared 99.5th-percentile scale, and apply a 0.65 display gamma so mid-range cornering changes remain readable. The recurring 96.5278% low-speed sentinel is shown as unavailable rather than as wheelspin.
- Recordings whose exported lap remains zero can still use the fixed reference track, position dot, speed, controls, power, g-meter, state of charge, and thermal overlay.
- The filename offset is only an initial synchronization estimate. Confirm `SYNC_ADJUST_S` visually before a full render.